In [1]:
import os
import rasterio
from rasterio.enums import Resampling
import numpy as np

In [ ]:
def merge_sentinel2_bands(jp2_folder, output_tiff_path, target_resolution=10):
    """
    Merge Sentinel-2 band files (.jp2) into a single .tif file with resampling and extended band descriptions.
    
    Args:
        jp2_folder (str): Path to the folder containing .jp2 files.
        output_tiff_path (str): Path to the output .tif file.
        target_resolution (int): Target resolution (in meters) for all bands (default: 10m).
    
    Returns:
        None
    """
    # Order of bands
    band_order = [
        "B01", "B02", "B03", "B04", "B05", "B06", "B07",
        "B08", "B8A", "B09", "B10", "B11", "B12"
    ]
    band_descriptions = {
        "B01": "B1", "B02": "B2", "B03": "B3", "B04": "B4",
        "B05": "B5", "B06": "B6", "B07": "B7", "B08": "B8",
        "B8A": "B8A", "B09": "B9", "B10": "B10",
        "B11": "B11", "B12": "B12"
    }
    
    # List of file .jp2 in the folder
    jp2_files = [f for f in os.listdir(jp2_folder) if f.endswith('.jp2')]
    
    # Filter and sort
    sorted_files = []
    valid_bands = []
    for band in band_order:
        matching_files = [os.path.join(jp2_folder, f) for f in jp2_files if band in f]
        if matching_files:
            sorted_files.append(matching_files[0])
            valid_bands.append(band)  # Just add decent bands
        else:
            print(f"Band {band} not found. Skipping...")

    if not sorted_files:
        print("No valid bands found in the folder. Exiting...")
        return

    # Metadata from the first band
    with rasterio.open(sorted_files[0]) as src:
        dst_transform = src.transform
        dst_crs = src.crs
        dst_width = int((src.bounds.right - src.bounds.left) / target_resolution)
        dst_height = int((src.bounds.top - src.bounds.bottom) / target_resolution)
    
    # Resample all the bands
    band_arrays = []
    for jp2_file in sorted_files:
        with rasterio.open(jp2_file) as src:
            band_data = src.read(
                1,
                out_shape=(dst_height, dst_width),
                resampling=Resampling.bilinear
            )
            band_arrays.append(band_data)

    # Merge bands to array 3D
    stacked_bands = np.stack(band_arrays, axis=0)  # Shape: (n_valid_bands, height, width)

    # Write file .tif
    with rasterio.open(
        output_tiff_path,
        "w",
        driver="GTiff",
        height=dst_height,
        width=dst_width,
        count=len(valid_bands),
        dtype=stacked_bands.dtype,
        crs=dst_crs,
        transform=dst_transform,
    ) as dst:
        for band_idx, band_data in enumerate(stacked_bands):
            dst.write(band_data, band_idx + 1)
            # Add description
            band_name = valid_bands[band_idx]
            dst.set_band_description(band_idx + 1, band_descriptions[band_name])

    print(f"Sentinel-2 bands merged into {output_tiff_path}")

In [ ]:
jp2_folder = "C:\\Users\\thanh\\Downloads\\T48QWJ_20240914T032511" # folder bands
output_tiff = "C:\\Users\\thanh\\Downloads\\Ouput_IMG\T48QWJ_20240914T032511.tif" # Save .tif

In [12]:
merge_sentinel2_bands(jp2_folder, output_tiff)

Band B01 not found. Skipping...
Band B05 not found. Skipping...
Band B06 not found. Skipping...
Band B07 not found. Skipping...
Band B8A not found. Skipping...
Band B09 not found. Skipping...
Band B10 not found. Skipping...
Band B11 not found. Skipping...
Band B12 not found. Skipping...
Sentinel-2 bands merged into C:\Users\thanh\Downloads\Ouput_IMG\T48QWJ_20240914T032511.tif
